In [ ]:
# CLEANING & TRANSATION

## User Data Cleaning & Transformation Decisions

- We fixed timestamp formats to make them readable and usable for analysis.
- We kept `user_id` as it is because it is a unique identifier.
- Country values were standardized to clear names.
- Email can be NULL, and this is acceptable in the system.
- Last login can also be NULL if the user never logged in.
- We created simple flags:
  - `has_email` (1 if email exists, 0 if NULL)
  - `has_logged_in` (1 if last_login exists, 0 if NULL)
- We found some invalid cases: if `last_login < created_at`, we treat it as invalid data and mark it using a `last_login_validated` with an additional flag `is_login_brfore_creation`.
- We mark it using a flag instead of deleting the row.
- We also added a review flag for edge cases like email being NULL but last_login existing.
- We did not delete any records, only cleaned and marked issues.
- The goal was to keep raw data safe and create a clean version for analysis.

In [5]:
dfu.createOrReplaceTempView("users_table")
dfs.createOrReplaceTempView("subscriptions_table")
dfp.createOrReplaceTempView("payments_table")

In [14]:
spark.sql("""
    SELECT
        CASE
            WHEN country = 'KSA' THEN 'Saudi Arabia'
            WHEN country = 'EGY' THEN 'Egypt'
            WHEN country = 'UAE' THEN 'United Arab Emirates'
            WHEN country = 'QAT' THEN 'Qatar'
            WHEN country = 'KWT' THEN 'kuwait'
            WHEN country = 'JOR' THEN 'Jordan'
            WHEN country = 'MAR' THEN 'morocco'
            ELSE 'Unkown'
        END AS country_name,
        country
    FROM users_table
    """
).show(5)

+------------+-------+
|country_name|country|
+------------+-------+
|     morocco|    MAR|
|Saudi Arabia|    KSA|
|      kuwait|    KWT|
|Saudi Arabia|    KSA|
|      kuwait|    KWT|
+------------+-------+
only showing top 5 rows



In [21]:
spark.sql("""
     WITH converted_timestamp AS (
        SELECT
            CAST(created_at / 1000000000 AS TIMESTAMP) AS created_at_timestamp,
            CAST(last_login / 1000000000 AS TIMESTAMP) AS last_login_timestamp
        FROM users_table
    )
    SELECT
        created_at,
        last_login,
        CAST(created_at AS DATE) AS created_date,
        CAST(last_login AS DATE) AS last_login_date
    FROM converted_timestamp
        
""").show(5)

+--------------------+--------------------+------------+---------------+
|          created_at|          last_login|created_date|last_login_date|
+--------------------+--------------------+------------+---------------+
|2025-08-21 13:43:...|                NULL|  2025-08-21|           NULL|
|2025-04-05 15:33:...|2025-04-29 15:33:...|  2025-04-05|     2025-04-29|
|2026-01-06 17:12:...|2026-01-12 17:12:...|  2026-01-06|     2026-01-12|
|2025-12-22 21:22:...|2025-12-30 21:22:...|  2025-12-22|     2025-12-30|
|2025-12-08 22:15:...|2025-12-10 22:15:...|  2025-12-08|     2025-12-10|
+--------------------+--------------------+------------+---------------+
only showing top 5 rows



In [20]:
spark.sql("""
    SELECT
        user_id,
        email,
         CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END AS has_email,
         CASE WHEN last_login IS NOT NULL THEN 1 ELSE 0 END AS has_logged_in,
         CASE WHEN last_login < created_at THEN NULL ELSE last_login END AS last_login_validated,
         CASE WHEN last_login < created_at THEN 1 ELSE 0 END AS is_login_before_creation,
         CASE WHEN email IS NULL AND last_login IS NOT NULL THEN "needs_review" ELSE NULL
         END AS review_status
    FROM users_table
    """
).show(5)

+--------------------+--------------------+---------+-------------+-------------------+-----------------------+-----------+
|             user_id|               email|has_email|has_logged_in|   last_login_clean|last_login_invalid_flag|review_flag|
+--------------------+--------------------+---------+-------------+-------------------+-----------------------+-----------+
|55a042c8-70fb-48e...| jesse86@example.net|        1|            0|               NULL|                      0|       NULL|
|47b4d3ca-7832-439...|taylorrenee@examp...|        1|            1|1745940797681172000|                      0|       NULL|
|73298054-91af-41a...|clarkchristina@ex...|        1|            1|1768237922681172000|                      0|       NULL|
|792c5481-f58b-491...|emilywilson@examp...|        1|            1|1767129764681172000|                      0|       NULL|
|2b2a2ffe-e8a5-49d...|samantha83@exampl...|        1|            1|1765404934681172000|                      0|       NULL|
+-------

In [6]:
user_silver_transformation= """
WITH users_clean AS (
    SELECT
        user_id,
        email,
        COALESCE(
            MAP('KSA', 'Saudi Arabia',
                'ECY', 'Egypt',
                'UAE', 'United Arab Emirates',
                'QAT', 'Qatar',
                'KWT', 'Kuwait',
                'jOR', 'Jordan',
                'MAR', 'Morocco'
            )[country],
            'Unknown'
        ) AS country_name,
        account_status,
        CAST(created_at / 1000000000 AS TIMESTAMP) AS created_at_timestamp,
        CAST(last_login / 1000000000 AS TIMESTAMP) AS last_login_timestamp
    FROM users_table
)
SELECT *,
    CASE WHEN last_login_timestamp < created_at_timestamp THEN NULL ELSE last_login_timestamp END AS last_login_validated,
    CASE WHEN last_login_timestamp < created_at_timestamp THEN 1 ELSE 0 END AS is_login_before_creation,
    CASE WHEN email IS NULL AND last_login_timestamp IS NOT NULL THEN "needs_review" ELSE NULL
    END AS review_status,
    CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END AS has_email,
    CASE WHEN last_login_timestamp IS NOT NULL THEN 1 ELSE 0 END AS has_logged_in,
    CAST(created_at_timestamp AS DATE) AS created_date,
    CAST(last_login_timestamp AS DATE) AS last_login_date
FROM users_clean
"""

In [7]:
users_df = spark.sql(user_silver_transformation)

In [17]:
users_df.show(5,truncate=False,vertical=True)

-RECORD 0--------------------------------------------------------
 user_id                  | 55a042c8-70fb-48e7-bbab-fca7a7628dde 
 email                    | jesse86@example.net                  
 country_name             | Morocco                              
 account_status           | deleted                              
 created_at_timestamp     | 2025-08-21 13:43:53.681172           
 last_login_timestamp     | NULL                                 
 last_login_validated     | NULL                                 
 is_login_before_creation | 0                                    
 review_status            | NULL                                 
 has_email                | 1                                    
 has_logged_in            | 0                                    
 created_date             | 2025-08-21                           
 last_login_date          | NULL                                 
-RECORD 1--------------------------------------------------------
 user_id  

In [ ]:
# Subscriptions

## Subscription Data Cleaning & Transformation Decisions

- Subscription ID was kept as it is because it is the main identifier.
- Plan type values were reviewed and kept because no abnormal values were found.
- Date fields were cleaned by converting timestamps into readable date formats for analysis.
- Handled invalid subscription dates:
  - If `end_date < start_date`, the value was considered invalid and handled accordingly.
  - Added `is_end_date_before_start_date` to identify invalid date cases.
- Added `end_date_validated` to keep the cleaned end date after handling invalid cases.
- `is_active` values were reviewed and kept because no abnormal values were found.
- Created derived date fields to make analysis easier.
- Added `review_status` for records where `end_date` exists but is earlier than `start_date`.
- No records were deleted; data quality issues were handled using cleaning logic and flags.

In [ ]:
spark.sql("""
     WITH converted_timestamp AS (
        SELECT
            CAST(start_date / 1000000000 AS TIMESTAMP) AS start_date_timestamp,
            CAST(end_date / 1000000000 AS TIMESTAMP) AS end_date_timestamp
        FROM subscriptions_table
    )
    SELECT
        start_date,
        end_date,
        CAST(start_date AS DATE) AS start_date,
        CAST(end_date AS DATE) AS end_date
    FROM converted_timestamp
        
""").show(5)

In [ ]:
spark.sql("""
    SELECT
        user_id,
          CASE WHEN end_date < start_date THEN NULL ELSE end_date END AS end_date_validated,
          CASE WHEN end_date < start_date THEN 1 ELSE 0 END AS is_end_date_before_start_date,
          CASE WHEN end_date IS NOT NULL AND end_date < start_date THEN "needs_review" ELSE NULL END AS review_status 
    FROM subscriptions_table
    """
).show(5)

In [8]:
subscription_silver_transformation= """
WITH sub_clean AS (    
    SELECT
    subscription_id,
    user_id,
    plan_type,
    CAST(start_date / 1000000000 AS TIMESTAMP) AS start_date_timestamp,
    CAST(end_date / 1000000000 AS TIMESTAMP) AS end_date_timestamp,
    is_active,
    auto_renew
    FROM subscriptions_table
)
SELECT *,
    CASE WHEN end_date_timestamp < start_date_timestamp THEN NULL
    ELSE end_date_timestamp END AS end_date_validated,
    CASE WHEN end_date_timestamp < start_date_timestamp THEN 1 
    ELSE 0 END AS is_end_date_before_start_date,
    CASE WHEN end_date_timestamp IS NOT NULL AND end_date_timestamp < start_date_timestamp THEN "needs_review"
    ELSE NULL END AS review_status,
    CAST(start_date_timestamp AS DATE) AS start_date,
    CAST(end_date_timestamp AS DATE) AS end_date
FROM sub_clean
"""

In [9]:
subscriptions_df = spark.sql(subscription_silver_transformation)

In [31]:
subscriptions_df.show(5,truncate=False,vertical=True)

-RECORD 0-------------------------------------------------------------
 subscription_id               | 1e51e8b6-223d-4ac3-aa20-ccd3da41d90d 
 user_id                       | 55a042c8-70fb-48e7-bbab-fca7a7628dde 
 plan_type                     | premium                              
 start_date_timestamp          | 2025-10-20 16:01:45                  
 end_date_timestamp            | 2026-01-18 16:01:45                  
 is_active                     | false                                
 auto_renew                    | true                                 
 end_date_validated            | 2026-01-18 16:01:45                  
 is_end_date_before_start_date | 0                                    
 review_status                 | NULL                                 
 start_date                    | 2025-10-20                           
 end_date                      | 2026-01-18                           
-RECORD 1-------------------------------------------------------------
 subsc

In [ ]:
# payments

## Payments Data Cleaning & Transformation Decisions

- Payment IDs were kept as they are because they are unique identifiers, although they have a long UUID format.

- Timestamp fields were converted from nanoseconds to readable timestamp format for analysis.

- No NULL values were found in the payments table, so no NULL handling was required.

- Amount values were standardized based on payment status:
  - `success` payments keep a positive amount using absolute value.
  - `failed` payments are set to 0 because the transaction was not completed.
  - `refunded` payments are represented as negative amounts to reflect returned money.

- Payment dates were created in two formats:
  - `payment_date_timestamp` for detailed timestamp analysis.
  - `payment_date` for date-level analysis and easier aggregation.

- Original payment status was kept because it is important for analysis and reporting.

- No records were deleted; only values were cleaned and standardized while preserving the original data meaning.

In [27]:
spark.sql("""
    WITH converted_timestamp AS (
        SELECT
            CAST(payment_date / 1000000000 AS TIMESTAMP) AS payment_date_timestamp,
            CAST(payment_date_timestamp AS DATE) AS payment_date
        FROM payments_table
    )
    SELECT 
       payment_date_timestamp,
       payment_date
    FROM converted_timestamp
       
""").show(5)

+----------------------+------------+
|payment_date_timestamp|payment_date|
+----------------------+------------+
|   2025-10-20 16:01:45|  2025-10-20|
|   2026-02-14 11:41:56|  2026-02-14|
|   2025-07-28 00:58:11|  2025-07-28|
|   2025-11-18 02:40:28|  2025-11-18|
|   2025-08-23 16:07:07|  2025-08-23|
+----------------------+------------+
only showing top 5 rows



In [40]:
spark.sql("""
SELECT    
    CASE
         WHEN payment_status = "success" THEN ABS(amount)
         WHEN payment_status = "failed" THEN 0
         WHEN payment_status = "refunded" THEN -ABS(amount)
    END AS amount,
    payment_status
FROM payments_table
""").show(20)

+------+--------------+
|amount|payment_status|
+------+--------------+
|-29.99|      refunded|
| 29.99|       success|
| 14.99|       success|
| 29.99|       success|
|  9.99|       success|
| 29.99|       success|
| 14.99|       success|
| 29.99|       success|
| 14.99|       success|
| 29.99|       success|
|  9.99|       success|
|   0.0|        failed|
| 29.99|       success|
| 29.99|       success|
|   0.0|        failed|
|  9.99|       success|
|  9.99|       success|
| 14.99|       success|
|  9.99|       success|
|  9.99|       success|
+------+--------------+
only showing top 20 rows



In [10]:
payments_silver_transformation= """
SELECT
    payment_id,
    subscription_id,
    CASE
        WHEN payment_status = "success" THEN ABS(amount)
        WHEN payment_status = "failed" THEN 0
        WHEN payment_status = "refunded" THEN -ABS(amount)
    END AS amount,
    currency,
    CAST(payment_date / 1000000000 AS TIMESTAMP) AS payment_date_timestamp,
    CAST(CAST(payment_date/ 1000000000 AS TIMESTAMP)AS DATE) AS payment_date,
    payment_status
FROM payments_table
"""

In [11]:
payments_df = spark.sql(payments_silver_transformation)

In [34]:
payments_df.show(5,truncate=False,vertical=True)

-RECORD 0------------------------------------------------------
 payment_id             | 2e2d403f-6abc-4022-8034-6b4fb15ffae3 
 subscription_id        | 1e51e8b6-223d-4ac3-aa20-ccd3da41d90d 
 amount                 | -29.99                               
 currency               | USD                                  
 payment_date_timestamp | 2025-10-20 16:01:45                  
 payment_date           | 2025-10-20                           
 payment_status         | refunded                             
-RECORD 1------------------------------------------------------
 payment_id             | df38bca8-b67a-427c-976f-79b8a3bd4346 
 subscription_id        | db85095a-8c69-4864-90e1-99b2ba242edf 
 amount                 | 29.99                                
 currency               | USD                                  
 payment_date_timestamp | 2026-02-14 11:41:56                  
 payment_date           | 2026-02-14                           
 payment_status         | success       